In [1]:
print("hello")

hello


In [18]:
from unsloth import FastLanguageModel
import torch
import json
import random
import os

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [19]:
model_name = 'unsloth/Phi-3-mini-4k-instruct-bnb-4bit'

In [20]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.1.4: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 5070 Ti. Num GPUs = 1. Max memory: 15.447 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
FastLanguageModel.for_inference(model)

In [24]:
categories = {
    "Information Technology": "laptop won't turn on, building web, building application, building software, wifi broken, server crash, screen, printer offline, install software, recover files, virus removal, coding help, password reset",
    "Cooking": "private chef needed, baking cakes, meal prep for week, catering for party, vegan menu, teaching how to cook, food delivery for event, baking lessons, dietary restrictions",
    "Handiworks": "assemble furniture, fix door handle, hang shelves, painting wall, repair fence, fix roof leak, drywall patch, carpentry, broken window, lock replacement",
    "Plumbing": "sink leaking, toilet clogged, low water pressure, pipe burst, install faucet, shower not draining, water heater broken, blocked drain, leak detection",
    "Electricity": "outlet not working, light switch broken, breaker keeps tripping, install ceiling fan, rewire house, flickering lights, install EV charger, fuse blown",
    "Cleaning": "deep clean apartment, carpet cleaning, window washing, move-out cleaning, office cleaning, power wash driveway, tidy up messy room, laundry service",
    "Education": "math tutor, piano lessons, learn spanish, sat prep, help with history homework, physics tutor, coding lessons, essay writing help, english teacher",
    "Well Being": "yoga instructor, meditation coach, life coaching, stress management, mindfulness session, spiritual guidance, relaxation techniques, personal mentor",
    "Health": "nursing care for elderly, feel sick, need medicine, physiotherapy, wound dressing, post-surgery care, check blood pressure, medical assistance, home nurse, injection service",
    "Accounting": "file taxes, bookkeeping for small business, audit assistance, financial planning, payroll help, quickbooks support, tax return, expense tracking"
}

In [25]:
generation_prompt = """You are a synthetic data generator. Generate 60 unique, diverse, and realistic user job requests for the category: "{category}".

Context keywords to inspire you: {keywords}

Rules:
1. The requests must sound like REAL customers (some angry, some polite, some urgent, some short, some detailed).
2. Do NOT number the list.
3. Output ONLY the requests, one per line.
4. Do not start lines with "I need" every time. Vary the phrasing.
5. Do not use the category name in the request (e.g., don't say "I need plumbing", say "my pipe burst").
"""

In [26]:
data = []

print(f"Starting generation for {len(categories)} categories...")

for cat, keywords in categories.items():
    print(f"--> Generating data for: {cat}...")

    for i in range(2): #the request ask LLM to generate 50 query once, twice means 100
        messages = [
            {"role": "user", "content": generation_prompt.format(category=cat, keywords=keywords)}
        ]
        inputs = tokenizer.apply_chat_template(
            messages, 
            tokenize=True, 
            add_generation_prompt=True, 
            return_tensors="pt"
        ).to("cuda")

        outputs = model.generate(
            inputs, 
            max_new_tokens=2048,
            temperature=0.9, # High temperature = more creativity/variety
            use_cache=True
        )
        
        # Decode output
        decoded_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        
        # Extract the assistant's response (handling different Llama response formats)
        if "assistant" in decoded_text:
            raw_response = decoded_text.split("assistant")[-1].strip()
        else:
            raw_response = decoded_text
            
        lines = raw_response.split("\n")
        
        # Clean and filter lines
        count = 0
        for line in lines:
            line = line.strip()
            # Remove bullets, numbers, and garbage
            clean_line = line.lstrip("- ").lstrip("* ").lstrip("1234567890. ").strip()
            
            # Keep only valid-looking lines
            if len(clean_line) > 10 and not clean_line.startswith("Here is") and not clean_line.startswith("Sure"):
                data.append({"input": clean_line, "output": cat})
                count += 1
        
        print(f"    Batch {i+1}: Generated {count} examples")

# 5. SAVE TO FILE (JSONL format)
print(f"Shuffling and saving {len(data)} examples...")
random.shuffle(data)

Starting generation for 10 categories...
--> Generating data for: Information Technology...
    Batch 1: Generated 30 examples
    Batch 2: Generated 30 examples
--> Generating data for: Cooking...
    Batch 1: Generated 89 examples
    Batch 2: Generated 89 examples
--> Generating data for: Handiworks...
    Batch 1: Generated 118 examples
    Batch 2: Generated 118 examples
--> Generating data for: Plumbing...
    Batch 1: Generated 69 examples
    Batch 2: Generated 69 examples
--> Generating data for: Electricity...
    Batch 1: Generated 67 examples
    Batch 2: Generated 67 examples
--> Generating data for: Cleaning...
    Batch 1: Generated 78 examples
    Batch 2: Generated 78 examples
--> Generating data for: Education...
    Batch 1: Generated 131 examples
    Batch 2: Generated 131 examples
--> Generating data for: Well Being...
    Batch 1: Generated 121 examples
    Batch 2: Generated 121 examples
--> Generating data for: Health...
    Batch 1: Generated 79 examples
    Ba

In [42]:
with open("train3.jsonl", "w", encoding='utf-8') as f:
    for entry in data:
        json.dump(entry, f)
        f.write("\n")

print("Generating training data. Done")

Generating training data. Done


In [30]:
import json
import re

input_file = "train3.jsonl"
output_file = "train3_c.jsonl"


with open(input_file, 'r') as f_in, open(output_file, 'w') as f_out:
    for line in f_in:
        try:
            data = json.loads(line)
            data['input'] = data['input'].replace('\"', '') #clean /"
            data['input'] = re.sub(r'0{5,}', '', data['input']) #clean repeat character 
            f_out.write(json.dumps(data) + '\n')
        except json.JSONDecodeError:
            continue

print("Done")

Done


In [ ]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [43]:
print("Loading training dataset")
df = pd.read_json("train3_c.jsonl", lines=True)

Loading training dataset


In [45]:
# TF-IDF: Converts text to math
# linear svc: The best algorithm for text classification
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words="english", 
        max_features=1000,
        ngram_range=(1,1),
        dtype='float32')
    ),
    ("clf", LinearSVC(class_weight='balanced')),
])

In [46]:
# 3. Train
print("2. Training Model...")
X_train, X_test, y_train, y_test = train_test_split(df['input'], df['output'], test_size=0.1, random_state=42)
pipeline.fit(X_train, y_train)

# 4. Test
print("3. Validating...")
accuracy = pipeline.score(X_test, y_test)
print(f"Accuracy: {accuracy:.2f}")
print(classification_report(y_test, pipeline.predict(X_test)))

2. Training Model...
3. Validating...
Accuracy: 0.95
                        precision    recall  f1-score   support

            Accounting       1.00      0.92      0.96        12
              Cleaning       1.00      1.00      1.00        10
               Cooking       1.00      0.93      0.97        15
             Education       1.00      1.00      1.00        33
           Electricity       1.00      0.78      0.88         9
            Handiworks       1.00      0.88      0.93        32
                Health       1.00      1.00      1.00        18
Information Technology       0.60      1.00      0.75         6
              Plumbing       0.88      1.00      0.93        14
            Well Being       0.91      1.00      0.95        21

              accuracy                           0.95       170
             macro avg       0.94      0.95      0.94       170
          weighted avg       0.96      0.95      0.95       170



/home/will/anaconda3/envs/unsloth_env/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


In [ ]:
# 5. Save
joblib.dump(pipeline, "service_classifier.pkl")
print("Model saved")

Model saved to 'service_classifier.pkl' (Size: < 10MB)


In [ ]:
my_text = "I need help with baking cakes"
prediction = pipeline.predict([my_text])
print(f"input: '{my_text}'")
print(f"predicted: {prediction[0]}") # [0] gets the string out of the array

Input: 'I need help with baking cakes'
Predicted: Cooking


In [54]:
#predict from saved model

model = joblib.load("service_classifier.pkl")
user_input = "my sink is broken"
predict = model.predict([user_input])[0]

print(f"Category: {prediction}")

Category: ['Cooking']
